# Vault Kubernetes Auth (IAM Role/IRSA) 설명
HashiCorp 공식 문서 - https://developer.hashicorp.com/vault/docs/auth/aws

이 노트북은 Vault의 AWS 인증 방식을 Kubernetes 환경(EKS)에서 사용하는 방법을 다룹니다. 특히 IRSA(IAM Roles for Service Accounts)를 활용하여 Pod이 Vault에 인증하는 과정을 설명합니다.

1. **샘플 Application 생성**
2. **Vault AWS Auth 설정**
3. **Kubernetes 리소스 생성 (VSO 연동)**
4. **리소스 삭제 (Cleanup)**

## 환경변수 설정

In [ ]:
# direnv 환경변수 로드
direnv allow
eval $(direnv export bash)

unset AWS_ACCESS_KEY_ID
unset AWS_SECRET_ACCESS_KEY
unset AWS_SESSION_TOKEN

export SERVICE_NAME="sample-app2"
export NAMESPACE="sample-app2-ns"
export VAULT_NAMESPACE="vault"

# Vault 설정
export VAULT_AUTH_PATH="${SERVICE_NAME}-iam-auth"
export VAULT_ROLE_NAME="${SERVICE_NAME}-role"
export VAULT_POLICY_NAME="${SERVICE_NAME}-policy"

# 연동할 KV 설정
export KV_MOUNT_PATH="my-kv"
export KV_PATH="app/credentials"

# K8s 설정
export K8S_HOST=$(kubectl config view --minify --output 'jsonpath={.clusters[0].cluster.server}')
export SERVICE_ACCOUNT_NAME="${SERVICE_NAME}-sa"

# AWS 설정
export APP_IRSA_ROLE_ARN="arn:aws:iam::341689148868:role/sample-app2-vault-irsa-role"

echo "K8S_HOST: ${K8S_HOST}"
echo "NAMESPACE: ${NAMESPACE}"
echo "VAULT_AUTH_PATH: ${VAULT_AUTH_PATH}"
echo "VAULT_ROLE_NAME: ${VAULT_ROLE_NAME}"
echo "VAULT_POLICY_NAME: ${VAULT_POLICY_NAME}"
echo "SERVICE_ACCOUNT_NAME: ${SERVICE_ACCOUNT_NAME}"
echo "APP_IRSA_ROLE_ARN: ${APP_IRSA_ROLE_ARN}"

## 1. 샘플 Application 생성

In [ ]:
# Namespace 생성
kubectl create namespace ${NAMESPACE}

# App 이 사용할 ServiceAccount 생성
kubectl apply -f - <<EOF
apiVersion: v1
kind: ServiceAccount
metadata:
  name: ${SERVICE_ACCOUNT_NAME}
  namespace: ${NAMESPACE}
  annotations:
    eks.amazonaws.com/role-arn: ${APP_IRSA_ROLE_ARN}
EOF

# sample-app 생성
kubectl apply -f - <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ${SERVICE_NAME}
  namespace: ${NAMESPACE}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: ${SERVICE_NAME}
  template:
    metadata:
      labels:
        app: ${SERVICE_NAME}
    spec:
      serviceAccountName: ${SERVICE_ACCOUNT_NAME}
      containers:
      - name: main
        image: alpine
        command: ["sh", "-c", "while true; do ls -l /etc/secrets; cat /etc/secrets/*; sleep 10; done"]
        volumeMounts:
        - name: secret-volume
          mountPath: /etc/secrets
          readOnly: true
      volumes:
      - name: secret-volume
        secret:
          secretName: ${SERVICE_NAME}-k8s-secret
EOF

## 2. Vault AWS Auth 설정
Vault 에 k8s 인증이 아닌 AWS 인증을 활성화 하고, 필요한 설정을 진행합니다.

In [ ]:
# Role 에 할당할 Policy 생성 (KV 읽기 전용)
vault policy write sample-app2-policy - <<EOF
path "${KV_MOUNT_PATH}/data/${KV_PATH}" {
  capabilities = ["read"]
}
EOF

In [ ]:
# AWS Auth Method 활성화
vault auth enable --path=${VAULT_AUTH_PATH} aws || true

# Vault Role 생성 (IAM 타입)
# bound_iam_principal_arn 에 지정된 Role 만 인증이 가능함
vault write auth/${VAULT_AUTH_PATH}/role/${VAULT_ROLE_NAME} \
    auth_type=iam \
    bound_iam_principal_arn="${APP_IRSA_ROLE_ARN}" \
    policies="${VAULT_POLICY_NAME}" \
    token_ttl="10h" \
    token_max_ttl="10d" \
    resolve_aws_unique_ids=false

## 3. Kubernetes 리소스 생성 (VSO 연동)

VSO가 IRSA를 통해 AWS STS에 인증을 요청하고, 받아온 Identity로 Vault에 로그인합니다.

In [ ]:
# VaultConnection - Endpoint 관리라고 생각하면 됨
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultConnection
metadata:
  name: vault-connection
  namespace: ${NAMESPACE}
spec:
  address: ${VAULT_ADDR}
EOF

# VaultAuth - Vault 인증 관리
# method 에 따라 인증 방식 변경 가능
# 여기서는 aws 이며, spec/aws 안에서 세부적인 설정 필요함
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuth
metadata:
  name: ${SERVICE_NAME}-aws-auth
  namespace: ${NAMESPACE}
spec:
  method: aws
  mount: ${VAULT_AUTH_PATH}
  aws:
    role: ${VAULT_ROLE_NAME}
    irsaServiceAccount: ${SERVICE_ACCOUNT_NAME}
  vaultConnectionRef: vault-connection
EOF

# VaultStaticSecret - 실제로 VSO 가 연동할 Secret 정보 관리
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  name: ${SERVICE_NAME}-secret
  namespace: ${NAMESPACE}
spec:
  vaultAuthRef: ${SERVICE_NAME}-aws-auth

  mount: ${KV_MOUNT_PATH}
  type: kv-v2
  path: ${KV_PATH}
  destination:
    name: ${SERVICE_NAME}-k8s-secret
    create: true
  refreshAfter: 30s

  rolloutRestartTargets:
  - kind: Deployment
    name: ${SERVICE_NAME}
EOF

## 4. 리소스 삭제 (Cleanup)

테스트를 위해 생성한 모든 리소스를 삭제합니다.

In [ ]:
# 1. Kubernetes 리소스 삭제
kubectl delete deployment ${SERVICE_NAME} -n ${NAMESPACE}
kubectl delete vaultstaticsecret ${SERVICE_NAME}-secret -n ${NAMESPACE}
kubectl delete vaultauth ${SERVICE_NAME}-aws-auth -n ${NAMESPACE}
kubectl delete vaultconnection vault-connection -n ${NAMESPACE}
kubectl delete serviceaccount ${SERVICE_ACCOUNT_NAME} -n ${NAMESPACE}
kubectl delete namespace ${NAMESPACE}

# 2. Vault Auth 설정 삭제
vault auth disable ${VAULT_AUTH_PATH}